# With R : Convert Seurat object (RDS file) to AnnData (for python analysis : ex Scanpy and SCVI

## Set the working directory

In [6]:
setwd("/workspaces/microsplit_DOL")

## Import R packages

In [1]:
# Fonction to convert

# Required packages
library(Seurat)
library(scCustomize)
library(anndata)
library(reticulate)


Loading required package: SeuratObject

Loading required package: sp


Attaching package: 'SeuratObject'


The following objects are masked from 'package:base':

    intersect, t




Error in .make_numeric_version(x, strict, .standard_regexps()$valid_numeric_version) : 
  invalid non-character version specification 'x' (type: double)


scCustomize v3.0.1
If you find the scCustomize useful please cite.
See 'samuel-marsh.github.io/scCustomize/articles/FAQ.html' for citation info.


Attaching package: 'anndata'


The following object is masked from 'package:SeuratObject':

    Layers




## Configure reticulate env to use scCustomize package (python implementation)

In [2]:
# Configure reticulate to use conda
message("Configuring reticulate to use conda...")

# Use the conda environment
use_condaenv("env_reticulate", required = TRUE)


# Verify Python configuration
message("Verifying Python configuration...")
message("Python version:", py_version())
message("Python path:", py_config()$python)

# check if scanpy, pandas and anndata are installed and available and print their versions
packages <- py_list_packages()
cat("scanpy installed:", "scanpy" %in% packages$package, "\n")
cat("pandas installed:", "pandas" %in% packages$package, "\n")
cat("anndata installed:", "anndata" %in% packages$package, "\n")




# check if anndata is available
reticulate::py_module_available(module = "anndata")


Configuring reticulate to use conda...

Verifying Python configuration...

Python version:

Python path:/opt/conda/envs/env_reticulate/bin/python



scanpy installed: TRUE 
pandas installed: TRUE 
anndata installed: TRUE 


[1] TRUE

In [3]:
convert_seurat_to_anndata <- function(input_file, assay_name = "RNA", main_layer = "counts") {
  # Load required packages
  if (!requireNamespace("Seurat", quietly = TRUE)) stop("The 'Seurat' package is required.")
  if (!requireNamespace("scCustomize", quietly = TRUE)) stop("The 'scCustomize' package is required.")
  if (!requireNamespace("anndata", quietly = TRUE)) stop("The 'anndata' package is required.")
  if (!requireNamespace("reticulate", quietly = TRUE)) stop("The 'reticulate' package is required.")

  message("Reading RDS file: ", input_file)

  # Check if file exists
  if (!file.exists(input_file)) stop("File not found: ", input_file)

  # Load the Seurat object
  seurat_object <- readRDS(input_file)

  # Check that the requested assay exists
  if (!assay_name %in% names(seurat_object@assays)) {
    stop("Assay '", assay_name, "' not found in the Seurat object.")
  }

  # Set default assay
  Seurat::DefaultAssay(seurat_object) <- assay_name

  # Extract metadata
  seurat_meta <- seurat_object@meta.data

  # Define output file path
  output_dir <- dirname(input_file)
  output_name <- tools::file_path_sans_ext(basename(input_file))
  output_file <- file.path(output_dir, paste0(output_name, ".h5ad"))

  message("Converting to AnnData...")
  message("Output file: ", output_file)

  # Convert using scCustomize
  scCustomize::as.anndata(
    x = seurat_object,
    file_path = output_dir,
    file_name = paste0(output_name, ".h5ad"),
    main_layer = main_layer,
    other_layers = NULL,
    obs_names = colnames(seurat_object),
    var_names = rownames(seurat_object[[assay_name]]),
    obs = seurat_meta
  )

  # Check if file was created
  if (!file.exists(output_file)) stop("Failed to write the .h5ad file.")

  message("Successfully converted to .h5ad: ", output_file)

  # Verify with initial read
  message("Verifying output .h5ad file...")
  adata <- anndata::read_h5ad(output_file)

  return(adata)
}


In [ ]:

convert_seurat_to_anndata(
  input_file = "jupyter/correct/correct_V2/ScObjects/SeuratObject_Val_Filtered_CDS_5cells.rds"
)

Reading RDS file: jupyter/correct/correct_V2/ScObjects/SeuratObject_Val_Filtered_CDS_5cells.rds

Converting to AnnData...

Output file: jupyter/correct/correct_V2/ScObjects/SeuratObject_Val_Filtered_CDS_5cells.h5ad

* Checking Seurat object validity
* Extracting Data from RNA assay to transfer to anndata.
* Creating anndata object.
* Writing anndata file:
  "/workspaces/microsplit_DOL/jupyter/correct/correct_V2/ScObjects/SeuratObject_Val_Filtered_CDS_5cells.h5ad"
Successfully converted to .h5ad: jupyter/correct/correct_V2/ScObjects/SeuratObject_Val_Filtered_CDS_5cells.h5ad

Verifying output .h5ad file...



AnnData object with n_obs <U+00D7> n_vars = 2934 <U+00D7> 5491
    obs: 'nCount_RNA', 'nFeature_RNA', 'barcode.rd1', 'barcode.rd2', 'barcode.rd3', 'Well_loc_rd1', 'Well_loc_rd2', 'Well_loc_rd3', 'Sample_ID', 'Culture_medium', 'Rep_bio', 'Rep_tech', 'OD_timepoint', 'OD_theoric', 'OD_measured', 'Condition', 'Condition_OD_timepoint', 'Rep', 'Condition_OD_timepoint_Rep_bio'
    var: 'names'

Attention les metadatas des gènes sont perdus lors de la conversion, il faut les ajouter manuellement dans le fichier AnnData (voir le script python original ou le notebook : Add metadata to AnnData)

```r